# BM25 Hyperparameter Tuning (Task 2.6)

Grid search `k1 ∈ {1.2, 1.5, 2.0}` và `b ∈ {0.5, 0.75, 1.0}` trên 30 dev queries.

In [ ]:
from pathlib import Path
import json
import sys
from typing import Any

def find_project_root(start: Path) -> Path:
    """Tìm thư mục gốc project.

    Input:
        start: thư mục hiện tại khi mở notebook.
    Output:
        Path thư mục gốc chứa TEAM_ASSIGNMENTS.md và data/.
    """
    for p in [start, *start.parents]:
        if (p / "TEAM_ASSIGNMENTS.md").exists() and (p / "data").exists():
            return p
    raise FileNotFoundError("Cannot find project root")

project_root = find_project_root(Path.cwd().resolve())
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from retrieval.bm25_retriever import build_bm25_index, bm25_retrieve

def load_eval_inputs(corpus_path: Path, queries_path: Path, qrels_path: Path) -> tuple[list[dict[str, Any]], list[dict[str, Any]], dict[int, set[int]]]:
    """Đọc dữ liệu tune.

    Input:
        corpus_path: file data/corpus.json
        queries_path: file data/queries.json
        qrels_path: file data/qrels.json
    Output:
        (corpus, queries, qrels_int_set)
    """
    corpus = json.loads(corpus_path.read_text(encoding="utf-8"))
    queries = json.loads(queries_path.read_text(encoding="utf-8"))
    raw_qrels = json.loads(qrels_path.read_text(encoding="utf-8"))
    qrels = {int(qid): {int(doc_id) for doc_id in doc_ids} for qid, doc_ids in raw_qrels.items()}
    return corpus, queries, qrels

def build_dev_queries(queries: list[dict[str, Any]], qrels: dict[int, set[int]], dev_size: int = 30) -> list[dict[str, Any]]:
    """Chọn 30 query dev có ground-truth.

    Input:
        queries: toàn bộ query list
        qrels: map query_id -> set doc_id liên quan
        dev_size: số query dev cần lấy
    Output:
        list query dev (sắp theo query_id tăng dần)
    """
    eligible = [q for q in queries if int(q["query_id"]) in qrels]
    eligible.sort(key=lambda q: int(q["query_id"]))
    return eligible[:dev_size]

def reciprocal_rank(ranked_doc_ids: list[int], relevant: set[int]) -> float:
    """Tính RR cho 1 query.

    Input:
        ranked_doc_ids: danh sách doc_id theo thứ hạng
        relevant: tập doc_id đúng theo qrels
    Output:
        1/rank hit đầu tiên, hoặc 0.0 nếu không có hit
    """
    if not relevant:
        return 0.0
    for rank, doc_id in enumerate(ranked_doc_ids, start=1):
        if doc_id in relevant:
            return 1.0 / rank
    return 0.0

def precision_at_k(ranked_doc_ids: list[int], relevant: set[int], k: int = 10) -> float:
    """Tính Precision@k cho 1 query.

    Input:
        ranked_doc_ids: danh sách doc_id theo thứ hạng
        relevant: tập doc_id đúng
        k: cutoff
    Output:
        tỉ lệ hit trong top-k
    """
    if k <= 0:
        return 0.0
    hits = sum(1 for doc_id in ranked_doc_ids[:k] if doc_id in relevant)
    return hits / k

def evaluate_params(corpus: list[dict[str, Any]], dev_queries: list[dict[str, Any]], qrels: dict[int, set[int]], k1: float, b: float, top_k: int = 100) -> dict[str, Any]:
    """Đánh giá một cặp tham số BM25.

    Input:
        corpus: dữ liệu tài liệu
        dev_queries: danh sách query dev
        qrels: ground-truth relevance
        k1, b: tham số BM25
        top_k: số doc retrieve/query
    Output:
        dict gồm k1, b, num_dev_queries, mrr, p@10
    """
    bm25, doc_ids, _ = build_bm25_index(
        corpus=corpus,
        k1=k1,
        b=b,
        include_title=True,
        use_stopwords=True,
        apply_stemming=False,
    )
    rr_scores = []
    p10_scores = []
    for q in dev_queries:
        qid = int(q["query_id"])
        ranked = bm25_retrieve(
            query=str(q["text"]),
            bm25=bm25,
            doc_ids=doc_ids,
            top_k=top_k,
            apply_stemming=False,
            use_stopwords=True,
        )
        ranked_doc_ids = [doc_id for doc_id, _ in ranked]
        relevant = qrels.get(qid, set())
        rr_scores.append(reciprocal_rank(ranked_doc_ids, relevant))
        p10_scores.append(precision_at_k(ranked_doc_ids, relevant, k=10))
    return {
        "k1": k1,
        "b": b,
        "num_dev_queries": len(dev_queries),
        "mrr": (sum(rr_scores) / len(rr_scores)) if rr_scores else 0.0,
        "p@10": (sum(p10_scores) / len(p10_scores)) if p10_scores else 0.0,
    }

def grid_search_bm25(corpus: list[dict[str, Any]], queries: list[dict[str, Any]], qrels: dict[int, set[int]], k1_values: list[float], b_values: list[float], dev_size: int = 30, top_k: int = 100) -> dict[str, Any]:
    """Grid search BM25 trên dev queries.

    Input:
        corpus/queries/qrels: dữ liệu đầu vào
        k1_values: danh sách k1 cần thử
        b_values: danh sách b cần thử
        dev_size: số query dev
        top_k: số lượng docs retrieve
    Output:
        dict chứa results đầy đủ và best params
    """
    dev_queries = build_dev_queries(queries, qrels, dev_size=dev_size)
    results = []
    for k1 in k1_values:
        for b in b_values:
            results.append(evaluate_params(corpus, dev_queries, qrels, k1=k1, b=b, top_k=top_k))
    results.sort(key=lambda x: (x["mrr"], x["p@10"]), reverse=True)
    return {
        "dev_size": len(dev_queries),
        "k1_values": k1_values,
        "b_values": b_values,
        "results": results,
        "best": results[0] if results else {},
    }

def save_tuning_results(results: dict[str, Any], output_path: Path) -> None:
    """Lưu kết quả tuning.

    Input:
        results: dict output từ grid_search_bm25
        output_path: file JSON cần lưu
    Output:
        Không return; ghi file ra đĩa.
    """
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text(json.dumps(results, ensure_ascii=False, indent=2), encoding="utf-8")

corpus_path = project_root / "data" / "corpus.json"
queries_path = project_root / "data" / "queries.json"
qrels_path = project_root / "data" / "qrels.json"
output_path = project_root / "reports" / "bm25_tuning_results.json"

corpus, queries, qrels = load_eval_inputs(corpus_path, queries_path, qrels_path)
print("project_root:", project_root)
print("corpus:", len(corpus), "queries:", len(queries), "qrels:", len(qrels))

In [ ]:
results = grid_search_bm25(
    corpus=corpus,
    queries=queries,
    qrels=qrels,
    k1_values=[1.2, 1.5, 2.0],
    b_values=[0.5, 0.75, 1.0],
    dev_size=30,
    top_k=100,
)
save_tuning_results(results, output_path)

print("Saved:", output_path)
print("Best:", results["best"])

In [ ]:
for row in results["results"]:
    print(f"k1={row['k1']}, b={row['b']} -> MRR={row['mrr']:.6f}, P@10={row['p@10']:.6f}")

In [ ]:
import matplotlib.pyplot as plt

k1_values = results["k1_values"]
b_values = results["b_values"]
score_map = {(r["k1"], r["b"]): r["mrr"] for r in results["results"]}
matrix = [[score_map[(k1, b)] for b in b_values] for k1 in k1_values]

fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(matrix, aspect="auto")
ax.set_xticks(range(len(b_values)))
ax.set_yticks(range(len(k1_values)))
ax.set_xticklabels([str(b) for b in b_values])
ax.set_yticklabels([str(k1) for k1 in k1_values])
ax.set_xlabel("b")
ax.set_ylabel("k1")
ax.set_title("BM25 tuning heatmap (MRR on 30 dev queries)")

for i, k1 in enumerate(k1_values):
    for j, b in enumerate(b_values):
        ax.text(j, i, f"{matrix[i][j]:.3f}", ha="center", va="center", color="white")

best = results["best"]
best_i = k1_values.index(best["k1"])
best_j = b_values.index(best["b"])
ax.scatter(best_j, best_i, s=180, facecolors="none", edgecolors="red", linewidths=2, label="Best")
ax.legend(loc="upper right")

fig.colorbar(im, ax=ax, label="MRR")
fig.tight_layout()
plt.show()